# Data Cleaning

Loads the raw Chicago Taxi 2024 dataset, cleans financial columns, parses timestamps, removes nulls and invalid rows, and caps outliers.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
df = pd.read_csv('chicago_taxi_trips.csv')
df.head(5)

/var/folders/1d/10_pykf97q78z_l8jtn_zf900000gn/T/ipykernel_58307/2949829491.py:1: DtypeWarning: Columns (0: Trip Miles) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('chicago_taxi_trips.csv')


,Trip ID,Taxi ID,Trip Start Timestamp,Trip End Timestamp,Trip Seconds,Trip Miles,Pickup Census Tract,Dropoff Census Tract,Pickup Community Area,Dropoff Community Area,...,Extras,Trip Total,Payment Type,Company,Pickup Centroid Latitude,Pickup Centroid Longitude,Pickup Centroid Location,Dropoff Centroid Latitude,Dropoff Centroid Longitude,Dropoff Centroid Location
0,993d8142f83842bb8246a33a9830bf0ef6b26fca,e3a458804fe9298cd1ab49287e64a8a8354d9a0766f062...,12/31/2024 11:45:00 PM,01/01/2025 12:45:00 AM,"3,060",0.0,NaN,NaN,28.0,32.0,...,$0.00,$16.00,Cash,Taxi Affiliation Services,41.874005,-87.663518,POINT (-87.6635175498 41.874005383),41.878866,-87.625192,POINT (-87.6251921424 41.8788655841)
1,cb8c2c403e0f9b664383842656170c256617c450,cb8da8dde1057540d8da485b4510d133316163e21fbef9...,12/31/2024 11:45:00 PM,01/01/2025 12:15:00 AM,"1,367",1.37,NaN,NaN,76.0,76.0,...,$0.00,$12.50,Cash,5 Star Taxi,41.980264,-87.913625,POINT (-87.913624596 41.9802643146),41.980264,-87.913625,POINT (-87.913624596 41.9802643146)
2,d5b465a9c5a1dda4a2800ae9a63dc19a12b1c160,062551a6a256e259d266eb4904910a6d2fe1b649bc7144...,12/31/2024 11:45:00 PM,01/01/2025 12:15:00 AM,"1,324",17.62,NaN,NaN,76.0,32.0,...,$4.00,$60.62,Credit Card,Sun Taxi,41.980264,-87.913625,POINT (-87.913624596 41.9802643146),41.878866,-87.625192,POINT (-87.6251921424 41.8788655841)
3,02918eed346c35e29a159a7b603a98b94e804389,c7a8a53874bbcdb11e70a488485e8bdd0bb8cc0de8f5d9...,12/31/2024 11:45:00 PM,01/01/2025 12:15:00 AM,"1,122",15.27,NaN,NaN,28.0,49.0,...,$0.00,$38.00,Prcard,5 Star Taxi,41.874005,-87.663518,POINT (-87.6635175498 41.874005383),41.706588,-87.623367,POINT (-87.6233665115 41.7065878819)
4,0986d2b51d18a8f7aa9bbabb701ca39e41ba06fd,342474dd3a24c9d5b7dbc2514032a01ecb396624638be5...,12/31/2024 11:45:00 PM,01/01/2025 12:00:00 AM,496,3.25,NaN,NaN,8.0,28.0,...,$0.00,$13.08,Mobile,Tac - American United Dispatch,41.899602,-87.633308,POINT (-87.6333080367 41.899602111),41.874005,-87.663518,POINT (-87.6635175498 41.874005383)


In [ ]:
# 1. Clean money-related columns
money_cols = ["Fare", "Tips", "Tolls", "Extras", "Trip Total"]

for col in money_cols:
    # Remove dollar signs and extra spaces
    df[col] = (
        df[col]
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.strip()
    )
    
    # Convert to numeric (invalid values → NaN)
    df[col] = pd.to_numeric(df[col], errors="coerce")


# 2. Clean numeric trip columns (remove commas, convert to numbers)
df["Trip Seconds"] = pd.to_numeric(
    df["Trip Seconds"].astype(str).str.replace(",", ""),
    errors="coerce"
)

df["Trip Miles"] = pd.to_numeric(
    df["Trip Miles"].astype(str).str.replace(",", ""),
    errors="coerce"
)


# 3. Convert timestamps to datetime
df["Trip Start Timestamp"] = pd.to_datetime(
    df["Trip Start Timestamp"], errors="coerce"
)
df["Trip End Timestamp"] = pd.to_datetime(
    df["Trip End Timestamp"], errors="coerce"
)


# 4. Convert community areas to nullable integers
df["Pickup Community Area"] = df["Pickup Community Area"].astype("Int64")
df["Dropoff Community Area"] = df["Dropoff Community Area"].astype("Int64")


# 5. Remove rows where all financial values are missing
df.dropna(
    subset=["Fare", "Tips", "Tolls", "Extras", "Trip Total"],
    how="all",
    inplace=True
)


# 6. Filter out invalid trips
df = df[
    (df["Trip Seconds"] > 0) &   # valid duration
    (df["Trip Miles"] >= 0) &    # valid distance
    (df["Fare"] > 0) &           # must have fare
    (df["Trip Total"] > 0)       # must have total
]


# 7. Standardize text columns
df["Payment Type"] = df["Payment Type"].str.strip().str.title()
df["Company"] = df["Company"].str.strip()


# 8. Create useful features (for analysis / Tableau)
df["Trip Minutes"] = (df["Trip Seconds"] / 60).round(2)
df["Trip Start Hour"] = df["Trip Start Timestamp"].dt.hour
df["Trip Start Date"] = df["Trip Start Timestamp"].dt.date
df["Trip Start Month"] = df["Trip Start Timestamp"].dt.to_period("M").astype(str)
df["Day of Week"] = df["Trip Start Timestamp"].dt.day_name()

# Binary flag for tipping behavior
df["Is Tipped"] = (df["Tips"] > 0).astype(int)

# Speed calculation (miles per hour)
df["Speed MPH"] = (
    df["Trip Miles"] / (df["Trip Seconds"] / 3600)
).round(2)

# Handle divide-by-zero issues
df["Speed MPH"] = df["Speed MPH"].replace([np.inf, -np.inf], np.nan)


# 9. Quick sanity check
print("Shape:", df.shape)
print("\nData Types:\n", df.dtypes)
print("\nMissing Values:\n", df.isnull().sum())

(6346900, 28)
Trip ID                                  str
Taxi ID                                  str
Trip Start Timestamp          datetime64[us]
Trip End Timestamp            datetime64[us]
Trip Seconds                         float64
Trip Miles                           float64
Pickup Census Tract                      str
Dropoff Census Tract                     str
Pickup Community Area                  Int64
Dropoff Community Area                 Int64
Fare                                 float64
Tips                                 float64
Tolls                                float64
Extras                               float64
Trip Total                           float64
Payment Type                             str
Company                                  str
Pickup Centroid Latitude             float64
Pickup Centroid Longitude            float64
Dropoff Centroid Latitude            float64
Dropoff Centroid Longitude           float64
Trip Minutes                         floa

In [10]:
df["Trip Start Date"] = df["Trip Start Date"].astype(str)